In [1]:
%pip install python-dotenv
%pip install roboflow
%pip install supervision

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   -- ------------------------------------- 2.6/38.8 MB 13.7 MB/s eta 0:00:03
   ------ --------------------------------- 6.0/38.8 MB 14.7 MB/s eta 0:00:03
   --------- ------------------------------ 8.9/38.8 MB 14.2 MB/s eta 0:00:03
   ------------ --------------------------- 12.6/38.8 MB 14.9 MB/s eta 0:00:02
   ---------------- ----------------------- 16.3/38.8 MB 15.0 MB/s eta 0:00:02
   -------------------- ------------------- 19.7/38.8 MB 15.5 MB/s eta 0:00:02
   ------------------------ --------------- 23.3/38.8 MB 15.7 MB/s eta 0:00:01
   --------------------------- ------------ 26.7/38.8 MB 15.7 MB/s eta 0:00:01
   ------------------------------- -------- 30.4/38.8 MB 15.7 MB/s eta 0:00:01
   ----------------------------------- ---- 34.1/38.8 MB 15.9 MB/s eta 0:00:01
   -------------------------------------- - 37.2/38.8 MB 15.9 MB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 12.1.0 which is incompatible.


INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   -- ------------------------------------- 2.9/40.2 MB 15.3 MB/s eta 0:00:03
   ------ --------------------------------- 6.3/40.2 MB 16.1 MB/s eta 0:00:03
   --------- ------------------------------ 10.0/40.2 MB 16.4 MB/s eta 0:00:02
   ------------ --------------------------- 12.3/40.2 MB 14.9 MB/s eta 0:00:02
   --------------- ------------------------ 16.0/40.2 MB 15.3 MB/s eta 0:00:02
   ------------------- -------------------- 19.4/40.2 MB 15.5 MB/s eta 0:00:02
   ---------------------- ----------------- 23.1/40.2 MB 15.7 MB/s eta 0:00:02
   -------------------------- ------------- 26.7/40.2 MB 15.8 MB/s eta 0:00:01
   ----------------------------- ---------- 30.1/40.2 MB 15.9 MB/s eta 0:00:01
   --------------------------------- ------ 33.8/40.2 MB 15.9 MB/s eta 0:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 12.1.0 which is incompatible.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getcwd())
load_dotenv(".env")

c:\Users\anony\Coding projects\Personal Projects\vitamin-tracking\lahari_himal


True

In [3]:
# loads dataset
from roboflow import Roboflow
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=os.path.join(os.getcwd(), ".env"))
print(os.getenv("TEST"))

api_key = os.getenv("YF_API_KEY")

rf = Roboflow(api_key=api_key)
project = rf.workspace("caretech").project("food-dataset-uj20h-w2s4m")
version = project.version(1)
dataset = version.download("yolov8")

test
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Food-Dataset-1 in yolov8:: 100%|██████████| 9832/9832 [00:20<00:00, 491.10it/s]


In [5]:
# script to split training dataset
import os
import shutil
from pathlib import Path


# creates a new folder with just the images and labels
dir = Path('/content/Food-Dataset-1')
unified_dir = Path('/content/all')
split_path = Path('/content/split')

new_img_dir = unified_dir / 'images'
new_label_dir = unified_dir / 'labels'

new_img_dir.mkdir(parents=True, exist_ok=True)
new_label_dir.mkdir(parents=True, exist_ok=True)

# write to a flattened folder
for file in dir.rglob('*.jpg'):
    shutil.copy(file, new_img_dir)

exclude = ['README.roboflow.txt', 'README.dataset.txt']
for file in dir.rglob('*.txt'):
    if str(file.name) in exclude:
        continue
    shutil.copy(file, new_label_dir)


In [8]:
import supervision as sv
# this loads a DetectionDataset object
ds = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{str(new_img_dir)}",
    annotations_directory_path=f"{str(new_label_dir.name)}",
    data_yaml_path=f"{str(dir.name)}/data.yaml"
)

print(ds.classes)

seed = 1
# we can split this dataset deterministically
train_ds, rest_ds = ds.split(split_ratio=0.8, random_state=seed, shuffle=True)
test_ds, val_ds = rest_ds.split(split_ratio=0.5, random_state=seed, shuffle=True)

# save new datasets in yolo format
train_ds.as_yolo(
    images_directory_path=str(split_path / 'train' / 'images'),
    annotations_directory_path=str(split_path / 'train' / 'labels'),
    data_yaml_path=str(split_path / 'train' / 'data.yaml')
)
test_ds.as_yolo(
    images_directory_path=str(split_path / 'test' / 'images'),
    annotations_directory_path=str(split_path / 'test' / 'labels'),
    data_yaml_path=str(split_path / 'test' / 'data.yaml')
)
val_ds.as_yolo(
    images_directory_path=str(split_path / 'valid' / 'images'),
    annotations_directory_path=str(split_path / 'valid' / 'labels'),
    data_yaml_path=str(split_path / 'valid' / 'data.yaml')
)

# write the manifest files
def write_manifest(ds: sv.DetectionDataset, split: str, output_path: Path):
  with open(output_path, "w") as f:
    for img_path, _, _ in ds:
      f.write(f"{str(Path(img_path).name)}\n")

  print(f"Wrote to {output_path}")

write_manifest(train_ds, "train", Path(split_path) / "train.txt")
write_manifest(test_ds, "test", Path(split_path) / "test.txt")
write_manifest(val_ds, "valid", Path(split_path) / "valid.txt")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [9]:
# class distribution count
from collections import defaultdict
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator

class_labels = { i: ds.classes[i] for i in range(len(ds.classes)) }

class_count = defaultdict(int)
box_count = defaultdict(int)
for file in unified_dir.rglob('*.txt'):
    count = 0
    with open(file, 'r') as f:
        for line in f:
          arr = line.split()
          if not arr:
            continue

          i = int(arr[0])
          class_count[class_labels[i]] += 1
          count += 1

    box_count[count] += 1




# class distribution count
# image size
# aspect ratio
# boxes per image
# noise / blur

# plot
plt.figure(figsize=(10, 6))
plt.bar(class_count.keys(), class_count.values())
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution')
plt.xticks(rotation=90)
plt.show()

# plot
plt.figure(figsize=(10, 6))
ax = plt.figure().gca()
plt.bar(box_count.keys(), box_count.values())
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_xlim([0, 10])
plt.xlabel('Box Count')
plt.ylabel('Count')
plt.title('Box Count Distribution')
plt.show()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\anony\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\anony\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\anony\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\anony\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: initialization failed

In [ ]:
from PIL import Image

# image size
sizes: list[tuple[int, int]] = [] # width x height
for img in unified_dir.rglob('*.jpg'):
  with Image.open(img) as im:
    sizes.append(im.size)

ratios = [s[0] / s[1] for s in sizes]

# plot
plt.figure(figsize=(10, 6))
plt.scatter([s[0] for s in sizes], [s[1] for s in sizes])
plt.xlabel('Width')
plt.ylabel('Height')
plt.title('Image Sizes')
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(ratios, bins=100, range=(0.5, 2.5))
plt.xlabel('Aspect Ratio')
plt.ylabel('Count')
plt.title('Aspect Ratio Distribution')
plt.show()